## Plot For Bool Q Dataset

In [ ]:
import re
from typing import Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from multi_llm_debate.analysis.correct_rate_by_round import (
    calculate_correct_rate_by_round,
)
from multi_llm_debate.analysis.calculate_task_accuracy import analyze_task_accuracy
from multi_llm_debate.run.bool_q.utils import extract_bool_answer

# ==========================
# 1. CONFIGURATION
# ==========================
DATA_PATH = Path("../output/bool_q/processed_data.csv")
bool_q_path = Path("../datasets/bool_q")
MODEL_DIR_PATH = Path("../data/bool_q")

# We only want directories whose parentheses-based digit sum == 6
TARGET_MODEL_COUNT = 11

# Maximum round number for your correct_rate_by_round function
MAX_ROUND_NUMBER = 10


# ==========================
# 2. HELPER FUNCTIONS
# ==========================
def get_total_model_count(dir_name: str) -> int:
    """Parses folder name to sum up the numeric values found in parentheses.

    Args:
        dir_name: Model directory name, e.g., 'llama3(3)+mistral(3)'

    Returns:
        Sum of all numbers found in parentheses
    """
    matches = re.findall(r"\((\d+)\)", dir_name)
    return sum(int(m) for m in matches)


def create_plot(
    accuracies_by_round: Dict[float, Dict[str, np.ndarray]], model_name: str
) -> None:
    """Plots lines for each accuracy value and metric type.

    Args:
        accuracies_by_round: Dictionary mapping accuracy values to a dict of
            metrics with their corresponding values.
        model_name: Name of the model for the plot title.
    """
    # Sort the dictionary items by the accuracy value (the dictionary key)
    sorted_items = sorted(accuracies_by_round.items(), key=lambda x: x[0])

    # Use a color map for different accuracy values
    accuracy_colors = plt.cm.get_cmap("tab20", len(sorted_items))

    # Fixed colors for the different metrics
    metric_color_map = {
        "absolute": "#1f77b4",  # Blue
        "majority": "#ff7f0e",  # Orange
        "majority_vote": "#2ca02c",  # Green - alternative name if needed
    }

    plt.figure(figsize=(10, 6))

    # Define line styles for different metrics
    line_styles = {
        "absolute": "-",  # solid line
        "majority": "--",  # dashed line
        "majority_vote": "--",  # dashed line (alternative name)
    }

    legend_handles = []

    # Plot a line for each unique accuracy value and metric
    for idx, (accuracy, metrics_dict) in enumerate(sorted_items):
        for metric_name, values in metrics_dict.items():
            # Skip empty arrays or None values
            if values is None or len(values) == 0:
                continue

            # Create x-axis values matching the length of values
            rounds = np.arange(len(values))

            # Only plot if both rounds and values have the same length
            if len(rounds) > 0 and len(rounds) == len(values):
                # Get color for this specific metric
                if metric_name in metric_color_map:
                    color = metric_color_map[metric_name]
                else:
                    # Fallback to accuracy-based color if metric not in map
                    color = accuracy_colors(idx)

                (line,) = plt.plot(
                    rounds,
                    values,
                    color=color,
                    linestyle=line_styles.get(metric_name, "-"),
                    linewidth=2,
                    label=f"Acc={accuracy:.2f} ({metric_name})",
                )
                legend_handles.append(line)

    # Title and labels
    plt.title(f"Accuracy by Round: {model_name}", pad=15)
    plt.xlabel("Round Number")
    plt.ylabel("Correct Rate")
    plt.grid(True, linestyle="--", alpha=0.7)

    if legend_handles:
        plt.legend(handles=legend_handles)

    # Set y-axis limits and ticks
    plt.ylim(0, 1)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xticks(range(min(11, MAX_ROUND_NUMBER + 1)))

    plt.tight_layout()
    plt.show()


def process_model(model_dir: Path) -> None:
    """Process model data and create visualizations.

    Args:
        model_dir: Path to the model directory containing debate data.

    This function:
    1) Analyzes accuracy
    2) Calculates correct rates for each unique accuracy value
    3) Prints the percentage of tasks for each accuracy value
    4) Plots the results for both absolute and majority vote metrics
    """
    model_name = model_dir.name
    print(f"\nProcessing model: {model_name}")

    # 1) Analyze accuracy
    result_df = analyze_task_accuracy(
        model_dir=model_dir,
        dataframe=df,
        extract_fn=extract_bool_answer,
    )

    # 2) Get all unique accuracy values from the result dataframe
    unique_accuracies = result_df["accuracy"].unique()

    # 3) Create a dictionary to store the metrics by round for each accuracy
    accuracies_by_round = {
        accuracy: {"absolute": None, "majority_vote": None}
        for accuracy in unique_accuracies
    }

    length = len(result_df)
    # 4) For each unique accuracy value, calculate metrics by round
    for accuracy in unique_accuracies:
        if accuracy < 0:
            continue

        # Filter tasks by accuracy
        filtered_df = result_df[result_df["accuracy"] == accuracy]

        # Calculate and print the percentage of tasks with this accuracy
        accuracy_percentage = (len(filtered_df) / length) * 100
        print(f"Accuracy = {accuracy:.2f}: {accuracy_percentage:.2f}% of total tasks")

        try:
            # Calculate correct rates for this accuracy
            cr_filtered_df = calculate_correct_rate_by_round(
                filtered_df,
                model_dir,
                max_round_number=MAX_ROUND_NUMBER,
                extract_func=extract_bool_answer,
            )

            # Check if we have results for the metrics
            absolute_rows = cr_filtered_df[cr_filtered_df["metric"] == "absolute"]
            if not absolute_rows.empty:
                accuracies_by_round[accuracy]["absolute"] = absolute_rows.iloc[
                    0, 2:
                ].values

            majority_rows = cr_filtered_df[cr_filtered_df["metric"] == "majority"]
            if not majority_rows.empty:
                accuracies_by_round[accuracy]["majority"] = majority_rows.iloc[
                    0, 2:
                ].values

        except Exception as e:
            print(f"Error processing accuracy {accuracy}: {e}")
            continue

    # 5) Create the plot
    create_plot(accuracies_by_round, model_name)


# ==========================
# 3. MAIN SCRIPT
# ==========================
if __name__ == "__main__":
    from multi_llm_debate.run.bool_q.utils import process_bool_q_df
    from multi_llm_debate.utils.download_dataset import load_save_dataset_df
    import os

    # Load the main data
    if not DATA_PATH.exists():
        os.makedirs(DATA_PATH.parent, exist_ok=True)
        dataset = load_save_dataset_df(
            dataset_name="google/boolq",
            dataset_path=bool_q_path,
            force_download=False,
        )
        df = process_bool_q_df(dataset)
        df.to_csv(DATA_PATH, index=False)
    else:
        df = pd.read_csv(DATA_PATH)

    print("DF columns:", df.columns)

    # Get all model directories
    all_model_dirs = list(MODEL_DIR_PATH.glob("*"))

    # Filter directories by total model count == TARGET_MODEL_COUNT
    filtered_model_dirs = [
        d for d in all_model_dirs if get_total_model_count(d.name) == TARGET_MODEL_COUNT
    ]

    print(f"Filtered directories (sum of parentheses == {TARGET_MODEL_COUNT}):")
    for d in filtered_model_dirs:
        print("  -", d.name)

    # Process each filtered directory
    for model_dir in filtered_model_dirs:
        process_model(model_dir)

In [ ]:
if __name__ == "__main__":
    from multi_llm_debate.run.bool_q.utils import extract_bool_answer
    from multi_llm_debate.analysis.utils import compare_bool
    import logging
    from multi_llm_debate.analysis.visualization import correct_rate_main
    from pathlib import Path

    # Set up logging
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    )
    logger = logging.getLogger(__name__)
    # Define paths
    DATA_PATH = Path("../output/bool_q/processed_data.csv")
    MODEL_DIR_PATH = Path("../data/bool_q/llama3(11)")
    OUTPUT_DIR = Path("../output/visualizations/bool_q")

    # Run the main function
    correct_rate_main(
        data_path=DATA_PATH,
        model_dir=MODEL_DIR_PATH,
        output_dir=OUTPUT_DIR,
        extract_func=extract_bool_answer,
        compare_func=compare_bool,
        max_rounds=10,
        show_plots=True,
    )

## Plot For JudgeBench

### Round Number Distribution

In [ ]:
from multi_llm_debate.analysis.utils import count_files_per_directory
from multi_llm_debate.analysis.visualization import plot_file_count_distribution


def main(model_dir_path: str) -> None:
    """Main function to analyze and visualize file count distribution.

    Args:
        model_dir_path: Path to the model directory containing data.
    """
    print(f"Analyzing files in: {model_dir_path}")

    # Get file count distribution
    distribution = count_files_per_directory(model_dir_path)

    # Display summary
    # if distribution:
    #     print("\nFile count distribution across directories:")
    #     print(f"{'Files':<10} {'Directories':>12}")
    #     print("-" * 22)
    #     for count, num_dirs in sorted(distribution.items()):
    #         print(f"{count:<10} {num_dirs:>12}")

    #     total_dirs = sum(distribution.values())
    #     print(f"\nTotal directories analyzed: {total_dirs}")
    #     print(f"Average files per directory: {sum(k*v for k,v in distribution.items())/total_dirs:.2f}")
    # else:
    #     print("No directory data found.")

    # Create visualization
    plot_file_count_distribution(distribution)


if __name__ == "__main__":
    model_dir = "../data/judge_bench/llama3(11)"
    main(model_dir)
    model_dir = "../data/judge_bench/gemma2:2b(11)"
    main(model_dir)

### Accuracy by Round

In [ ]:
if __name__ == "__main__":
    import os
    import logging
    from pathlib import Path

    from multi_llm_debate.analysis.visualization import correct_rate_main as main
    from multi_llm_debate.run.judge_bench.utils import (
        extract_caption_a_b_answer,
        compare_judge_bench_responses,
        load_judge_bench_dataset,
    )

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    )
    logger = logging.getLogger(__name__)

    df = load_judge_bench_dataset(dataset_path="../datasets/JudgeBench")
    os.makedirs("../output/judge_bench", exist_ok=True)
    df_path = Path("../output/judge_bench/processed_data.csv")
    df.to_csv(df_path, index=False)

    model_dir = Path("../data/judge_bench/llama3(11)")
    OUTPUT_DIR = Path("../output/visualizations/judge_bench")

    main(
        data_path=df_path,
        model_dir=model_dir,
        output_dir=OUTPUT_DIR,
        max_rounds=10,
        show_plots=True,  # Set to True if you want to see the plot window
        extract_func=extract_caption_a_b_answer,
        compare_func=compare_judge_bench_responses,
        model_config="llama3(11)",
    )
    model_dir = Path("../data/judge_bench/gemma2:2b(11)")
    main(
        data_path=df_path,
        model_dir=model_dir,
        output_dir=OUTPUT_DIR,
        max_rounds=10,
        show_plots=True,  # Set to True if you want to see the plot window
        extract_func=extract_caption_a_b_answer,
        compare_func=compare_judge_bench_responses,
        model_config="gemma2:2b(11)",
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from multi_llm_debate.analysis.correct_rate_by_round import (
    calculate_correct_rate_by_round,
)
from multi_llm_debate.analysis.calculate_task_accuracy import analyze_task_accuracy
from multi_llm_debate.run.judge_bench.utils import (
    extract_caption_a_b_answer,
    compare_judge_bench_responses,
)

# Maximum round number for your correct_rate_by_round function
MAX_ROUND_NUMBER = 10


def create_plot_majority_aggregated(
    aggregated_majority_by_round: np.ndarray, model_name: str
) -> None:
    """Plots the aggregated majority accuracy value by round.

    Args:
        aggregated_majority_by_round: Array of majority correct rates aggregated across all accuracies for each round.
        model_name: Name of the model for the plot title.
    """
    # Create x-axis values (round numbers)
    rounds = np.arange(len(aggregated_majority_by_round))

    # Plot the aggregated majority correct rate
    plt.figure(figsize=(10, 6))

    plt.plot(
        rounds,
        aggregated_majority_by_round,
        color="tab:blue",  # You can choose any color you like
        linestyle="-",  # Solid line for the aggregated majority
        linewidth=2,
        label=f"Aggregated Majority Correct Rate",
    )

    # Title and labels
    plt.title(f"Aggregated Majority Correct Rate by Round: {model_name}", pad=15)
    plt.xlabel("Round Number")
    plt.ylabel("Correct Rate")
    plt.grid(True, linestyle="--", alpha=0.7)

    plt.legend()

    # Set y-axis limits and ticks
    plt.ylim(0, 1)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xticks(range(min(11, MAX_ROUND_NUMBER + 1)))

    plt.tight_layout()
    plt.show()


def process_model_majority_aggregated(model_dir: Path) -> None:
    """Process model data and create visualizations for the aggregated majority correct rate.

    Args:
        model_dir: Path to the model directory containing debate data.

    This function:
    1) Analyzes accuracy
    2) Calculates majority correct rates for each unique accuracy value
    3) Aggregates the results for the majority correct rate across all accuracy levels
    4) Plots the results for the aggregated majority correct rate
    """
    model_name = model_dir.name
    print(f"\nProcessing model: {model_name}")

    # 1) Analyze accuracy
    result_df = analyze_task_accuracy(
        model_dir=model_dir,
        dataframe=df,
        extract_fn=extract_caption_a_b_answer,
        compare_func=compare_judge_bench_responses,
    )

    # 2) Get all unique accuracy values from the result dataframe
    unique_accuracies = result_df["accuracy"].unique()

    # 3) Initialize a list to store the majority values by round for each accuracy
    majority_by_rounds_list = []

    length = len(result_df)
    # 4) For each unique accuracy value, calculate majority correct rates by round
    for accuracy in unique_accuracies:
        if accuracy < 0:
            continue

        # Filter tasks by accuracy
        filtered_df = result_df[result_df["accuracy"] == accuracy]

        # Calculate and print the percentage of tasks with this accuracy
        accuracy_percentage = (len(filtered_df) / length) * 100
        print(f"Accuracy = {accuracy:.2f}: {accuracy_percentage:.2f}% of total tasks")

        try:
            # Calculate majority correct rates for this accuracy
            cr_filtered_df = calculate_correct_rate_by_round(
                filtered_df,
                model_dir,
                max_round_number=MAX_ROUND_NUMBER,
                extract_func=extract_caption_a_b_answer,
                compare_func=compare_judge_bench_responses,
            )

            # Check if we have results for the majority metric
            majority_rows = cr_filtered_df[cr_filtered_df["metric"] == "majority"]
            if not majority_rows.empty:
                majority_by_rounds_list.append(majority_rows.iloc[0, 2:].values)

        except Exception as e:
            print(f"Error processing accuracy {accuracy}: {e}")
            continue

    # 5) Aggregate majority values across all accuracies
    if majority_by_rounds_list:
        # Stack the lists vertically and calculate the mean across the rows (across all accuracies)
        aggregated_majority_by_round = np.mean(
            np.vstack(majority_by_rounds_list), axis=0
        )

        # 6) Create the plot for aggregated majority correct rates
        create_plot_majority_aggregated(aggregated_majority_by_round, model_name)


# ==========================
# 3. MAIN SCRIPT
# ==========================
if __name__ == "__main__":
    from multi_llm_debate.run.judge_bench.utils import load_judge_bench_dataset
    import os

    df = load_judge_bench_dataset(dataset_path="../datasets/JudgeBench")
    os.makedirs("../output/judge_bench", exist_ok=True)
    df_path = Path("../output/judge_bench/processed_data.csv")
    df.to_csv(df_path, index=False)

    model_dir = Path("../data/judge_bench/llama3(11)")
    OUTPUT_DIR = Path("../output/visualizations/judge_bench")

    process_model_majority_aggregated(model_dir)
    model_dir = Path("../data/judge_bench/gemma2:2b(11)")
    process_model_majority_aggregated(model_dir)

In [ ]:
from typing import Dict
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from multi_llm_debate.analysis.correct_rate_by_round import (
    calculate_correct_rate_by_round,
)
from multi_llm_debate.analysis.calculate_task_accuracy import analyze_task_accuracy
from multi_llm_debate.run.judge_bench.utils import (
    extract_caption_a_b_answer,
    compare_judge_bench_responses,
)

# Maximum round number for your correct_rate_by_round function
MAX_ROUND_NUMBER = 10


def create_plot_absolute_only(
    absolute_by_round: Dict[float, np.ndarray], model_name: str
) -> None:
    """Plots the absolute accuracy value by round.

    Args:
        absolute_by_round: Dictionary mapping accuracy values to absolute
            metrics by round.
        model_name: Name of the model for the plot title.
    """
    # Sort the dictionary items by the accuracy value (the dictionary key)
    sorted_items = sorted(absolute_by_round.items(), key=lambda x: x[0])

    # Use a color map for different accuracy values
    accuracy_colors = plt.cm.get_cmap("tab20", len(sorted_items))

    plt.figure(figsize=(10, 6))

    legend_handles = []

    # Plot a line for each unique accuracy value
    for idx, (accuracy, absolute_values) in enumerate(sorted_items):
        # Skip empty arrays or None values
        if absolute_values is None or len(absolute_values) == 0:
            continue

        # Create x-axis values matching the length of absolute_values
        rounds = np.arange(len(absolute_values))

        # Only plot if both rounds and values have the same length
        if len(rounds) > 0 and len(rounds) == len(absolute_values):
            # Get color for this specific accuracy value
            color = accuracy_colors(idx)

            (line,) = plt.plot(
                rounds,
                absolute_values,
                color=color,
                linestyle="-",  # Solid line for absolute
                linewidth=2,
                label=f"Acc={accuracy:.2f} (absolute)",
            )
            legend_handles.append(line)

    # Title and labels
    plt.title(f"Absolute Correct Rate by Round: {model_name}", pad=15)
    plt.xlabel("Round Number")
    plt.ylabel("Correct Rate")
    plt.grid(True, linestyle="--", alpha=0.7)

    if legend_handles:
        plt.legend(handles=legend_handles)

    # Set y-axis limits and ticks
    plt.ylim(0, 1)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xticks(range(min(11, MAX_ROUND_NUMBER + 1)))

    plt.tight_layout()
    plt.show()


def process_model_absolute_only(model_dir: Path) -> None:
    """Process model data and create visualizations for the absolute correct rate.

    Args:
        model_dir: Path to the model directory containing debate data.

    This function:
    1) Analyzes accuracy
    2) Calculates absolute correct rates for each unique accuracy value
    3) Plots the results for the absolute correct rate
    """
    model_name = model_dir.name
    print(f"\nProcessing model: {model_name}")

    # 1) Analyze accuracy
    result_df = analyze_task_accuracy(
        model_dir=model_dir,
        dataframe=df,
        extract_fn=extract_caption_a_b_answer,
        compare_func=compare_judge_bench_responses,
    )

    # 2) Get all unique accuracy values from the result dataframe
    unique_accuracies = result_df["accuracy"].unique()

    # 3) Create a dictionary to store the absolute metric by round for each accuracy
    absolute_by_round = {}

    length = len(result_df)
    # 4) For each unique accuracy value, calculate absolute correct rates by round
    for accuracy in unique_accuracies:
        if accuracy < 0:
            continue

        # Filter tasks by accuracy
        filtered_df = result_df[result_df["accuracy"] == accuracy]

        # Calculate and print the percentage of tasks with this accuracy
        accuracy_percentage = (len(filtered_df) / length) * 100
        print(f"Accuracy = {accuracy:.2f}: {accuracy_percentage:.2f}% of total tasks")

        try:
            # Calculate absolute correct rates for this accuracy
            cr_filtered_df = calculate_correct_rate_by_round(
                filtered_df,
                model_dir,
                max_round_number=MAX_ROUND_NUMBER,
                extract_func=extract_caption_a_b_answer,
                compare_func=compare_judge_bench_responses,
            )

            # Check if we have results for the absolute metric
            absolute_rows = cr_filtered_df[cr_filtered_df["metric"] == "absolute"]
            if not absolute_rows.empty:
                absolute_by_round[accuracy] = absolute_rows.iloc[0, 2:].values

        except Exception as e:
            print(f"Error processing accuracy {accuracy}: {e}")
            continue

    # 5 Create the plot for absolute correct rates
    create_plot_absolute_only(absolute_by_round, model_name)


# ==========================
# 3. MAIN SCRIPT
# ==========================
if __name__ == "__main__":
    from multi_llm_debate.run.judge_bench.utils import load_judge_bench_dataset
    import os

    df = load_judge_bench_dataset(dataset_path="../datasets/JudgeBench")
    os.makedirs("../output/judge_bench", exist_ok=True)
    df_path = Path("../output/judge_bench/processed_data.csv")
    df.to_csv(df_path, index=False)

    model_dir = Path("../data/judge_bench/llama3(11)")
    OUTPUT_DIR = Path("../output/visualizations/judge_bench")

    process_model_absolute_only(model_dir)
    model_dir = Path("../data/judge_bench/gemma2:2b(11)")
    process_model_absolute_only(model_dir)

In [ ]:
if __name__ == "__main__":
    from pathlib import Path
    from multi_llm_debate.run.judge_bench.utils import (
        compare_judge_bench_responses,
        extract_caption_a_b_answer,
    )
    from multi_llm_debate.distribution_model.visualize_model import run_visualization

    OUTPUT_DIR = Path("../output/visualizations/judge_bench")
    MAX_ROUNDS = None  # or an int

    # Analysis settings
    FIT_METHOD = "direct"  # "direct" or "em" optimization approach
    N_RESTARTS = 2  # Number of random restarts for more stable fitting
    ENFORCE_INCREASING = False  # Enforce non-decreasing expected success probability

    # Call the visualization pipeline function
    run_visualization(
        answers_csv_path=Path("../output/judge_bench/processed_data.csv"),
        debates_csv_path=Path("../data/judge_bench/llama3(11)/debate_rounds.csv"),
        output_dir=OUTPUT_DIR,
        max_rounds=MAX_ROUNDS,
        fitting_method=FIT_METHOD,
        n_restarts=N_RESTARTS,
        verbose=True,
        enforce_increasing_success=ENFORCE_INCREASING,
        extract_func=extract_caption_a_b_answer,
        compare_func=compare_judge_bench_responses,
        model_config="llama3(11)",
    )

    run_visualization(
        answers_csv_path=Path("../output/judge_bench/processed_data.csv"),
        debates_csv_path=Path("../data/judge_bench/gemma2:2b(11)/debate_rounds.csv"),
        output_dir=OUTPUT_DIR,
        max_rounds=MAX_ROUNDS,
        fitting_method=FIT_METHOD,
        n_restarts=N_RESTARTS,
        verbose=True,
        enforce_increasing_success=ENFORCE_INCREASING,
        extract_func=extract_caption_a_b_answer,
        compare_func=compare_judge_bench_responses,
        model_config="gemma2:2b(11)",
    )